In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import solve
from numpy.polynomial.legendre import leggauss
import torch
from scipy.linalg import eigh
from scipy.linalg import solve



In [ ]:

# single step scheme:   u^{n+1} = R(delta_T * A_h) * u^{n} + delta_t * SUM[P_i(delta_T * A_h) * f_h(t_n + delta_T * c_i)]
#                                      =                                        =                       =
# precomputed values:            #R_T_A_h_coarse/fine   ||                #P_i_sum_coarse/fine||#RHS_values_coarse/fine

# RHS_values_coarse : precomputed right-hand-side values required by the coarse propagator
# RHS_values_fine   : precomputed right-hand-side values required by the fine propagator
# R_T_A_h_coarse    : matrix R_C(Delta_T * A_h)
# R_T_A_h_fine      : matrix R_F(delta_t * A_h)
# P_i_sum_coarse    : matrices P_i(Delta_T A_h)
# P_i_sum_fine      : matrices P_i(delta_t A_h)
# CP                : function implementing one single-step propagation, for FP and CP
# T_intervall       : considered time interval
# N                 : total number of fine time steps
# J                 : number of fine steps per coarse interval
# K                 : number of Parareal iterations
# R_C, P_C, C_C     : stability function, P_i-functions and numbers C_i of the coarse propagator
# R_F, P_F, C_F     : corresponding quantities for the fine propagator
# A_h               : discrete operator
# v_h               : discrete initial value


def parareal_algorithm(RHS_values_coarse, RHS_values_fine, R_T_A_h_coarse, R_T_A_h_fine, P_i_sum_coarse, P_i_sum_fine, CP, T_intervall, M, N, J, u_0, f, R_C, R_F, P_C, P_F, C_C, C_F, K, A_h, stiff, mass, v_h):

  t_0, t_end = T_intervall
  T_n_coarse = np.linspace(t_0, t_end, int(N/J)+1)
  delta_T = (t_end - t_0)/(int(N/J))

  U = solve_on_grid(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, CP, T_intervall, v_h, f_h, int(N/J), R_C, P_C, C_C, A_h)
  for k in range(K):

    U_alt = U.copy()
    Fine_solutions = [v_h]

    for i in range(int(N/J)):
        U_n_j = U[i]
        U_n1_k = solve_on_grid(RHS_values_fine, R_T_A_h_fine, P_i_sum_fine, CP, (T_n_coarse[i],T_n_coarse[i+1]), U_n_j, f_h, J, R_F, P_F, C_F, A_h)[-1]
        Fine_solutions.append(U_n1_k)

    for i in range(int(N/J)):
      U[i+1] = CP(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, T_n_coarse[i], delta_T, U[i], f_h, R_C, P_C, C_C, A_h) + Fine_solutions[i+1] - CP(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, T_n_coarse[i], delta_T, U_alt[i], f_h, R_C, P_C, C_C, A_h)

  return U
